# Band structure and density of states of silicon

In this tutorial you take a Quantum ESPRESSO (QE) calculation of bulk silicon and turn it into a tight-binding Hamiltonian in a basis of pseudo-atomic orbitals (PAOs), which PAOFLOW generates from the pseudopotential and onto which it projects the Kohn–Sham wavefunctions. From this Hamiltonian you compute the band structure along the high-symmetry lines of the face-centred cubic (fcc) Brillouin zone, the density of states (DOS) on a k-mesh twice as dense as the DFT grid in each direction, and the Boltzmann transport tensors at 300 K. The sequence of calls, {py:meth}`~PAOFLOW.PAOFLOW.PAOFLOW.projections`, {py:meth}`~PAOFLOW.PAOFLOW.PAOFLOW.projectability`, {py:meth}`~PAOFLOW.PAOFLOW.PAOFLOW.pao_hamiltonian`, {py:meth}`~PAOFLOW.PAOFLOW.PAOFLOW.bands`, {py:meth}`~PAOFLOW.PAOFLOW.PAOFLOW.interpolated_hamiltonian`, {py:meth}`~PAOFLOW.PAOFLOW.PAOFLOW.pao_eigh`, {py:meth}`~PAOFLOW.PAOFLOW.PAOFLOW.gradient_and_momenta`, {py:meth}`~PAOFLOW.PAOFLOW.PAOFLOW.adaptive_smearing`, {py:meth}`~PAOFLOW.PAOFLOW.PAOFLOW.dos` and {py:meth}`~PAOFLOW.PAOFLOW.PAOFLOW.transport`, is the start of every PAOFLOW workflow, and the later tutorials begin from the Hamiltonian built here.

**What you will compute**

- The PAO basis of Si, generated from its pseudopotential, and the projections of the Kohn–Sham states onto it
- The bands this basis represents (the projectability)
- The real-space PAO Hamiltonian $H_{\alpha\beta}(\mathbf{R})$, obtained by Fourier transform of the projected Hamiltonian $\tilde H(\mathbf{k})$
- The band structure of Si along $\Gamma$–X–W–K–$\Gamma$–L–U–W–L–K and U–X, interpolated from a $12\times12\times12$ DFT grid
- The band velocities, and the adaptive Gaussian broadening built from them
- The DOS and its projection onto the Si $3s$, $3p$, $3d$, $4s$, $4p$ and $4d$ orbitals
- The conductivity, Seebeck coefficient and electronic thermal conductivity at constant relaxation time, as functions of the chemical potential

:::{tip}
This page is built on the case [`si_bands_transport`](https://github.com/marcobn/PAOFLOW/tree/develop/examples/cases/si_bands_transport): open it in PAOFLOW Studio with *Examples…* → `si_bands_transport`, or copy it with `paoflow-gen --example si_bands_transport`. Studio and `paoflow-gen` write the driver `main.py` that this notebook walks through, with the same PAOFLOW calls in the same order, so you can run the workflow as a script, for example on a cluster. To change its parameters, or start your own material from this workflow, {download}`download this notebook <tutorial01.ipynb>`. The input is {TODO: the case-data file of si_bands_transport and its release}. The first code cell of the notebook downloads it and extracts `silicon.save` next to the notebook; on a machine without internet access, download it by hand and extract it in the directory you run in. The notebook runs in {TODO: run time from run} on one core and needs {TODO: memory from run} of memory.
:::

## Required input

`silicon.save` is the output directory of two QE runs on bulk Si, whose input files are those of the case: `scf.in`, the self-consistent run, and `nscf.in`, the non-self-consistent run on the k-grid of this tutorial. PAOFLOW reads three kinds of file from it:

- `data-file-schema.xml`: the crystal structure, the k-point grid and the Kohn–Sham eigenvalues
- `wfc*.dat`: the Kohn–Sham wavefunctions, one file per k-point, which PAOFLOW projects onto the PAO basis
- `Si.pbe-tm-new-gipaw-v2.1.UPF`: the norm-conserving PBE pseudopotential, from which PAOFLOW generates the radial functions of the PAO basis

If you would like to create the `.save` directory yourself instead of downloading it, copy the case with `paoflow-gen --example si_bands_transport`, which puts the pseudopotential next to the inputs, and run the two inputs in order in its folder:

```bash
mpirun -np 4 pw.x -in scf.in > scf.out
mpirun -np 4 pw.x -in nscf.in > nscf.out
```

For your own material, start from these inputs.

To run this notebook on your own calculation, set `savedir` in the next cell to its `.save` directory; the download then does not run.

In [ ]:
from pathlib import Path
from tarfile import open as open_tar
from urllib.request import urlretrieve

savedir = 'silicon.save'
outputdir = 'output'

# fetch the DFT output of the case from the release named in the tip, unless it is here already
# TODO (PR 3 of docs-restructure-plan.md): the file and release of the case data of si_bands_transport
if not Path(savedir).is_dir():
    archive = Path('si_bands_transport-v1.tar.gz')
    if not archive.is_file():
        urlretrieve(
            'https://github.com/marcobn/PAOFLOW/releases/download/'
            'case-data/si_bands_transport-v1.tar.gz',
            archive,
        )
    with open_tar(archive, 'r:gz') as tar:
        members = [m for m in tar.getmembers() if m.name.startswith('silicon.save/')]
        tar.extractall(members=members, filter='data')

## Projecting the Kohn–Sham states onto the PAO basis

The PAO basis $\{\phi_\alpha\}$ consists of radial functions of the isolated Si pseudo-atom times spherical harmonics, and you first write these radial functions to disk. `generate_basis_for_pseudo()` takes the shells that the pseudopotential file stores as pseudo-orbitals, $3s$, $3p$ and $3d$ for Si, as they are, and computes the other shells as bound states of the pseudo-atom in a spherical box. It writes one file per shell to `BASIS_PS/Si/`. The radial functions depend only on the pseudopotential, so the cell makes them once, for the `extended` preset, the largest, which contains the shells of every other preset; the generation takes a few minutes, and the cell skips it when the folder exists.

In [ ]:
from PAOFLOW.basis_gen import generate_basis_for_pseudo

basispath = 'BASIS_PS/'
if not Path(basispath, 'Si').is_dir():
    generate_basis_for_pseudo(
        f'{savedir}/Si.pbe-tm-new-gipaw-v2.1.UPF', 'BASIS_PS', preset='extended', verbose=True
    )

The `PAOFLOW` constructor reads `data-file-schema.xml` and prints a summary of the calculation. `projections()` then builds the basis of the `standard` preset from the files in `basispath`: the $3s$, $3p$ and $3d$ pseudo-orbitals, and for each of them the shell one principal quantum number higher, $4s$, $4p$ and $4d$, which gives the conduction states more radial freedom. That is 18 PAOs per atom, so the two-atom cell has $n_\mathrm{awf} = 36$ PAOs (the presets and the source of each shell are described in [Internal basis workflow](../../internals/internal-basis.md)). From the wavefunctions, `projections()` computes the projection amplitudes $A_{n\alpha}(\mathbf{k}) = \langle\phi_\alpha|\psi_{n\mathbf{k}}\rangle$ of every Kohn–Sham state onto Löwdin-orthogonalised orbitals, so the basis is orthonormal and the overlap matrix $S(\mathbf{k})$ does not enter the PAO Hamiltonian built below (see [Löwdin orthogonalisation](../theory/pao-hamiltonian.md#löwdin-orthogonalisation) for what the orthogonalisation changes).

The other arguments of the constructor set how the run proceeds. With `smearing='gauss'` the DOS and the transport computed below use Gaussian broadening. `npool=1` keeps the k-points in one pool; on a larger system, more pools distribute the k-points across MPI ranks and reduce the memory per process. `save_overlaps=False` discards the overlap matrices of the projections once the Hamiltonian is built, because only ACBN0 needs them, and `verbose=False` limits the output to the summary and the time of each step.

In [ ]:
from PAOFLOW import PAOFLOW

paoflow = PAOFLOW.PAOFLOW(
    savedir=savedir,
    outputdir=outputdir,
    smearing='gauss',
    npool=1,
    save_overlaps=False,
    verbose=False,
)
paoflow.projections(basispath=basispath, configuration='standard')

The summary repeats the settings of the input files: an insulator with 8 electrons, 18 bands and 72 irreducible k-points of the $12\times12\times12$ grid. Because the occupations are fixed, the DFT Fermi energy $E_F$ of {TODO: DFT Fermi energy from run} eV is the highest occupied level, the valence-band maximum (VBM), and PAOFLOW takes it as the zero of all energies below.

:::{tip}
`projwfc.x` of QE computes the projections onto the pseudo-orbitals of the pseudopotential alone, the $3s$, $3p$ and $3d$ shells of Si, nine PAOs per atom. To use them, run `projwfc.x` after `nscf.in` and replace the basis generation and `projections()` by

```python
paoflow.read_atomic_proj_QE()
```

which reads the amplitudes from `atomic_proj.xml` in the `.save` directory. In Studio, choose the projector *projwfc* in the form of the case, which adds the `projwfc.x` run.
:::

## Projectability: which bands the PAO basis can represent

A Kohn–Sham state enters the PAO Hamiltonian only if the basis reproduces it. The projectability, the norm of the state's projection onto the PAOs, measures how much of the state the basis reproduces:

$$
p_{n\mathbf{k}} = \sum_\alpha |A_{n\alpha}(\mathbf{k})|^2 .
$$

A state built from Si $3s$ and $3p$ orbitals has $p_{n\mathbf{k}} \approx 1$, and a state far above the conduction-band minimum (CBM), with plane-wave character between the atoms, a smaller one. `projectability()` computes the average $\bar p_n$ of each band over the 72 k-points of the DFT run and counts the bands with $\bar p_n$ above `pthr`, here the default 0.95; the Hamiltonian is built from that number of lowest bands ([Number of kept bands](../theory/pao-hamiltonian.md#number-of-kept-bands)). This count needs bands above those the basis represents, so the DFT run computed 18 bands, 4 occupied and 14 empty; when every computed band passes the threshold, PAOFLOW prints a warning, and the run needs a larger `nbnd`. `projectability()` also sets the PAO shift $\eta$, the energy at which the PAO Hamiltonian places the part of the PAO space that the kept states do not span. With the default `shift='auto'`, $\eta$ is the lowest energy over the grid of the first band beyond the kept ones, so below $\eta$ the PAO bands follow the DFT bands ([The PAO shift](../theory/pao-hamiltonian.md#the-pao-shift)).

In [ ]:
paoflow.projectability(pthr=0.95)

{TODO: the projectability of the valence and conduction bands, the number of bands above pthr and which bands are kept, and the PAO shift $\eta$ with the energy range of the first band beyond the kept ones, from a run with verbose=True}

:::{tip}
With `verbose=True` in the constructor, `projectability()` prints $\bar p_n$ for every band, the number of bands above `pthr`, and the energy range of the first band beyond the kept ones, inside which any manual choice of `shift` should lie.
:::

## The real-space PAO Hamiltonian

At each k-point of the DFT grid, `pao_hamiltonian()` builds an $n_\mathrm{awf}\times n_\mathrm{awf}$ Hamiltonian $\tilde H(\mathbf{k})$ from the kept states: their eigenvalues on the part of the PAO space they span, and the energy $\eta$ on the rest, above the bands you can trust (the default `shift_type=1`; the formula is in [The PAO Hamiltonian at the DFT k-points](../theory/pao-hamiltonian.md#the-pao-hamiltonian-at-the-dft-k-points)). At each k-point, a kept band is dropped if its eigenvalue lies above $\eta$ or its $p_{n\mathbf{k}}$ is below half of `pthr`.

QE reduced the $12\times12\times12$ grid by symmetry to 72 irreducible points, and `pao_hamiltonian()` unfolds them onto the full grid of $N_k = 1728$ points ([From the irreducible wedge to the full grid](../theory/pao-hamiltonian.md#from-the-irreducible-wedge-to-the-full-grid)). The grid must be unshifted, because the discrete Fourier transform assumes points at the fractions $m_i/N_i$ of the reciprocal lattice vectors. A discrete Fourier transform over the $N_k$ grid points then gives the Hamiltonian in real space, $H_{\alpha\beta}(\mathbf{R})$, on a $12\times12\times12$ set of lattice vectors $\mathbf{R}$. $H_{\alpha\beta}(\mathbf{R})$ is the hopping between orbital $\alpha$ in the home cell and orbital $\beta$ in the cell at $\mathbf{R}$, an *ab initio* tight-binding model. Because the PAOs are localised, the hoppings decay with $|\mathbf{R}|$, so the Fourier sum of $H(\mathbf{R})$ evaluated at any $\mathbf{k}$ gives the bands between the grid points at the cost of a $36\times36$ diagonalisation ([The real-space Hamiltonian](../theory/pao-hamiltonian.md#the-real-space-hamiltonian)).

In [ ]:
paoflow.pao_hamiltonian()

## Band structure along the high-symmetry path

`bands()` evaluates $\tilde H(\mathbf{k})$ along a path through the Brillouin zone and diagonalises it at each point. `ibrav=2` is the QE index of the fcc lattice and selects the standard fcc path $\Gamma$–X–W–K–$\Gamma$–L–U–W–L–K followed by U–X (the label K|U marks the jump between the two pieces); `nk=2000` is the number of points along the whole path, enough to locate the CBM on the $\Gamma$–X line to within 0.3% of its length, and `fname='bands'` is the prefix of the output file.

**Output:**

- `bands_0.dat`: one row per k-point, the point index followed by the 36 band energies in eV relative to $E_F$
- `kpath_points.txt`: the high-symmetry points of the path and the number of points on each segment

In [ ]:
paoflow.bands(ibrav=2, nk=2000, fname='bands')

:::{tip}
To follow another path, give its labels and the crystal coordinates of its points; with `fname='bands_custom'` the result goes to `bands_custom_0.dat`, next to the default path.

```python
paoflow.bands(
    ibrav=2,
    band_path='L-G-X-W-K-G',
    high_sym_points={
        'G': [0.0, 0.0, 0.0],
        'L': [0.5, 0.5, 0.5],
        'X': [0.5, 0.5, 0.0],
        'W': [0.5, 0.25, 0.75],
        'K': [0.375, 0.375, 0.75],
    },
    nk=2000,
    fname='bands_custom',
)
```
:::

In [ ]:
import itertools
import xml.etree.ElementTree as ET
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from PAOFLOW import GPAO

output = Path('output')
hartree = 27.211386245988  # eV

# DFT eigenvalues on the irreducible k-points (Cartesian, 2pi/a), relative to E_F
root = ET.parse('silicon.save/data-file-schema.xml').getroot()
e_fermi = float(root.find('.//highestOccupiedLevel').text) * hartree
k_dft, e_dft = [], []
for ks in root.iter('ks_energies'):
    k_dft.append([float(x) for x in ks.find('k_point').text.split()])
    e_dft.append([float(x) * hartree - e_fermi for x in ks.find('eigenvalues').text.split()])
k_dft, e_dft = np.array(k_dft), np.array(e_dft)
nkept = 9  # TODO (PR 3): the number of kept bands of the run, from projectability()
eta = e_dft[:, nkept].min()  # PAO shift: lowest energy of the first band beyond them

# path points (Cartesian, 2pi/a) are listed after the labels in kpath_points.txt
k_path = np.loadtxt(output / 'kpath_points.txt', skiprows=13)
ops = [
    np.diag(s)[list(p)]
    for p in itertools.permutations(range(3))
    for s in itertools.product((1, -1), repeat=3)
]
on_path = []  # (index on the path, DFT k-point) for every grid point that lies on the path
for ik, k in enumerate(k_dft):
    star = np.array([op @ k for op in ops])
    hits = np.where(np.min(np.linalg.norm(k_path[:, None] - star[None], axis=2), axis=1) < 1e-4)[0]
    on_path += [(i, ik) for i in hits]

plotter = GPAO.GPAO()
ax = plotter.plot_bands(
    str(output / 'bands_0.dat'),
    sym_points=str(output / 'kpath_points.txt'),
    title='Si, PAO bands and DFT eigenvalues',
    label=r'$E - E_F$ (eV)',
    y_lim=(-12.0, 8.5),
    cols='#004488',
    show=False,
)
for i, ik in on_path:
    ax.scatter(np.full(nkept, i), e_dft[ik, :nkept], s=12, color='#D55E00', zorder=3)
ax.axhline(eta, color='#8A6800', ls='--', lw=1.0, label=rf'PAO shift $\eta$ = {eta:.2f} eV')
ax.scatter([], [], s=12, color='#D55E00', label=f'DFT eigenvalues, bands 1 to {nkept}')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.08), ncol=2, frameon=False)
plt.show()

The energy zero is $E_F$, the VBM. The valence band is {TODO: valence band width from run} eV wide. Its maximum at $\Gamma$ is threefold degenerate, as the cubic symmetry requires for the bonding $p$-like states. The CBM lies on the $\Gamma$–X line at {TODO: position of the CBM from run} of the distance to X, at {TODO: CBM from run} eV, so Si has an indirect gap of {TODO: gap from run} eV. This is the PBE gap of this calculation and not the measured one, because PBE eigenvalues underestimate band gaps.

The vermillion points are the DFT eigenvalues of the kept bands at the 21 points of the $12\times12\times12$ grid that lie on the path, read from `data-file-schema.xml`; the dashed line is the PAO shift $\eta$, the lowest energy of the first band beyond the kept ones on the grid. They give two checks of the PAO bands. First, at the grid points the PAO bands pass through the DFT eigenvalues: {TODO: agreement at $\Gamma$ of the two lowest conduction levels, and the lowest DFT conduction state on the grid next to the interpolated CBM, from run}. Away from $\Gamma$ the conduction bands agree less closely than the valence bands, because their normalised projections are not exactly orthogonal and the construction of the PAO Hamiltonian mixes them ([Accuracy at the DFT k-points](../theory/pao-hamiltonian.md#accuracy-at-the-dft-k-points)). Second, nothing at or above $\eta$ is a DFT band. The PAO bands there, {TODO: the bands at $\eta$ and their energy range, from run}, are the part of the PAO space that no kept state spans, and DFT points of kept bands above $\eta$ have no PAO band through them, because `pao_hamiltonian()` drops a kept state at a k-point where its energy lies above $\eta$. Results that depend on states near or above $\eta$ need a larger basis or a larger shift.

## Fourier interpolation onto a denser k-mesh

A Brillouin-zone integral such as the DOS needs a denser mesh than the $12\times12\times12$ DFT grid. Instead of a new DFT calculation, `interpolated_hamiltonian()` evaluates the Fourier sum of $H(\mathbf{R})$ on a mesh of `nfft1` $\times$ `nfft2` $\times$ `nfft3` points, here $24\times24\times24$, twice the DFT grid in each direction ($N_\mathrm{mesh} = 13824$), by zero-padding $H(\mathbf{R})$ before the fast Fourier transform ([Paths and meshes](../theory/pao-hamiltonian.md#paths-and-meshes)). `pao_eigh()` then diagonalises $\tilde H(\mathbf{k})$ at every mesh point and keeps the eigenvalues $\varepsilon_{n\mathbf{k}}$ and eigenvectors for the following steps. The memory of these steps grows with $N_\mathrm{mesh}\, n_\mathrm{awf}^2$; PAOFLOW prints its estimate before allocating the arrays.

In [ ]:
paoflow.interpolated_hamiltonian(nfft1=24, nfft2=24, nfft3=24)
paoflow.pao_eigh()

:::{tip}
Quantities that depend on a few states near a band edge, such as the carrier density of a doped semiconductor, need denser meshes, for example `interpolated_hamiltonian(nfft1=36, nfft2=36, nfft3=36)`; the [Boltzmann transport tutorial](tutorial02.ipynb) shows a convergence test.
:::

## Band velocities and adaptive smearing

`gradient_and_momenta()` computes the k-derivative of $\tilde H(\mathbf{k})$ analytically from $H(\mathbf{R})$, and its matrix elements between the eigenstates at each mesh point ([The gradient of the Hamiltonian](../theory/density-of-states.md#the-gradient-of-the-hamiltonian)). The diagonal elements are the band velocities $\mathbf{v}_{n\mathbf{k}} = \hbar^{-1}\nabla_{\mathbf{k}}\varepsilon_{n\mathbf{k}}$, which the [Boltzmann transport tutorial](tutorial02.ipynb) uses, and which here set the broadening of the DOS.

On a finite mesh each state stands for a small volume of the Brillouin zone, over which its energy changes by about $|\nabla_\mathbf{k}\varepsilon_{n\mathbf{k}}|\,\delta k$. `adaptive_smearing()` gives each state a Gaussian of a width proportional to that change (Yates *et al.*, Phys. Rev. B 75, 195121 (2007)):

$$
\sigma_{n\mathbf{k}} = a\,|\nabla_{\mathbf{k}}\varepsilon_{n\mathbf{k}}|\,\delta k ,
$$

where $\delta k = [8\pi^3/(\Omega N_\mathrm{mesh})]^{1/3}$ is the mean spacing of the mesh, $\Omega$ the cell volume, and $a = 0.7$ (the `afac` argument) for Gaussian smearing. Each width is the $\sigma$ of a normalised $e^{-x^2/\sigma^2}$, whose standard deviation is $\sigma/\sqrt{2}$ ([Width from the band velocity](../theory/density-of-states.md#width-from-the-band-velocity) derives the formula and the default of $a$). A state on a steep band gets a wide Gaussian that fills the gap to its neighbours on the mesh, and a state on a flat band gets a narrow one, so the DOS stays smooth without washing out sharp features. At a band extremum $\nabla_\mathbf{k}\varepsilon_{n\mathbf{k}} = 0$ and the width vanishes. A state much narrower than the spacing of the DOS energy grid is missed or counted as a spike, depending on where it falls on the grid; the check at the end of [Density of states and projected DOS](#density-of-states-and-projected-dos) shows whether this happened ([Band extrema, flat bands and the width floor](../theory/density-of-states.md#band-extrema-flat-bands-and-the-width-floor)).

In [ ]:
paoflow.gradient_and_momenta()
paoflow.adaptive_smearing()

:::{tip}
`adaptive_smearing(min_width=0.03)` sets a floor of 0.03 eV on the widths, about twice the 14 meV spacing of the DOS energy grid, so that no state falls between the grid points. Set such a floor when the DOS check finds states missing, and on meshes where the band velocity vanishes at many points, as on the flat bands of a model.
:::

## Density of states and projected DOS

`dos()` sums the Gaussians of all states on the mesh and divides by $N_\mathrm{mesh}$, which gives the DOS $g(\varepsilon)$ in states per eV, per spin and per cell ([Density of states on a k-mesh](../theory/density-of-states.md#density-of-states-on-a-k-mesh)). With `do_pdos=True` it also writes the projected DOS (PDOS) of each orbital $\alpha$, which weights each Gaussian with the orbital's share of the eigenvector, $|\langle\phi_\alpha|n\mathbf{k}\rangle|^2$, so below $\eta$, which covers the whole energy range of this call, the PDOS of all 36 orbitals adds up to $g(\varepsilon)$ ([Projected density of states](../theory/density-of-states.md#projected-density-of-states)). The call samples `ne=1000` energies from $-12$ to 2.2 eV, a 14 meV spacing, which covers the whole valence band and the bottom of the conduction band.

**Output:**

- `dosdk_0.dat`: energy (eV) and DOS (states/eV per spin and cell); the suffix `_0` is the spin channel
- `Si1_3s_pdosdk_0.dat`, `Si1_3px_pdosdk_0.dat`, ...: energy and PDOS, one file per PAO, named by atom and orbital
- `pdosdk_sum_0.dat`: energy and the sum of the PDOS over all orbitals

In [ ]:
paoflow.dos(do_dos=True, do_pdos=True, emin=-12.0, emax=2.2, ne=1000)

In [ ]:
from pathlib import Path

from PAOFLOW import GPAO

output = Path('output')
palette = ['#004488', '#D55E00', '#FFC20A', '#009E73', '#56B4E9', '#CC79A7']
plotter = GPAO.GPAO()
plotter.plot_dos(
    str(output / 'dosdk_0.dat'),
    title='Si, DOS per spin',
    x_lim=(-12.0, 2.2),
    col=palette[0],
)

# sum the PDOS of both atoms for each shell, read from the file names (Si1_3px_pdosdk_0.dat -> 3p)
pdos_files = sorted(output.glob('Si*_pdosdk_0.dat'))
shell = [f.name.split('_')[1][:2] for f in pdos_files]
shells = sorted(set(shell), key=lambda s: (s[0], 'spdf'.index(s[1])))
plotter.plot_pdos(
    [str(f) for f in pdos_files],
    title='Si, PDOS per spin',
    x_lim=(-12.0, 2.2),
    cols=palette[: len(shells)],
    labels=[f'Si {s}' for s in shells],
    psum_inds=[[i for i, s in enumerate(shell) if s == group] for group in shells],
)

The energy zero is the VBM. The DOS is zero between 0 and {TODO: CBM from run} eV, the indirect gap of the band structure. The highest peak of the valence DOS, at {TODO: energy of the highest valence peak from run} eV, comes from the flat second valence band around L in the band structure above. In the PDOS, {TODO: which shells carry the weight at the bottom of the valence band and near the VBM, from run}. Integrated over the valence band, the PDOS of the six shells hold {TODO: valence PDOS per shell from run} of the 4 states per spin. Si has no occupied $d$ shell, so the $d$ share is not an occupation: a PDOS decomposes the states in the chosen basis, and the Löwdin-orthogonalised $d$ PAOs overlap the bonding charge between the atoms.

The integral of the DOS over the valence band counts its states. Integrate the output file up to the middle of the gap:

In [ ]:
import numpy as np

energy, dos = np.loadtxt('output/dosdk_0.dat', unpack=True)
valence = energy < 0.377  # TODO (PR 3): half the gap of the run
print(f'valence states per spin: {np.trapezoid(dos[valence], energy[valence]):.6f}')

The result, {TODO: valence states per spin from run}, is the four valence bands per spin, or 8 electrons per cell with both spins. A value off by more than about $10^{-5}$ means that states were lost, narrower than the 14 meV spacing of the energy grid at the band extrema (set the floor of the tip in [Band velocities and adaptive smearing](#band-velocities-and-adaptive-smearing)), or that the energy range cuts off part of the band.

## Boltzmann transport at constant relaxation time

`transport()` computes the electrical conductivity $\sigma$, the Seebeck coefficient $S$ and the electronic thermal conductivity from the band velocities computed above, as functions of the chemical potential $\mu$, in the semiclassical Boltzmann theory ([Transport moments](../theory/boltzmann-transport.md#transport-moments)). Two approximations enter. Without scattering channels the relaxation time is the same for every state, $\tau = 1$ s, so the files hold $\sigma/\tau$, the thermal conductivity at zero electric field $\kappa_0/\tau$ and the power factor $S^2\sigma/\tau$, while $S$ does not depend on $\tau$ ([Constant relaxation time](../theory/boltzmann-transport.md#constant-relaxation-time)). The bands are rigid: changing $\mu$ moves the electrons through the bands of undoped Si, as if Si were doped without changing its electronic structure ([Rigid-band approximation](../theory/boltzmann-transport.md#rigid-band-approximation)). The call evaluates the tensors at one temperature, 300 K (`tmin`, `tmax`, `nt`), and at the 1000 values of $\mu$ of the DOS grid; only the kept bands of the projectability step contribute. `finish_execution()` then prints the total time and memory of the run.

**Output:** one file per tensor, with the columns temperature (K), $\mu$ relative to $E_F$ (eV), and the components $xx$, $yy$, $zz$, $xy$, $xz$, $yz$; the name carries the smearing, `gauss`.

- `sigmagauss_0.dat`: $\sigma/\tau$ in S m$^{-1}$ s$^{-1}$
- `Seebeckgauss_0.dat`: $S$ in V/K
- `kappagauss_0.dat`: $\kappa_0/\tau$ in W m$^{-1}$ K$^{-1}$ s$^{-1}$
- `PFgauss_0.dat`: $S^2\sigma/\tau$ in W m$^{-1}$ K$^{-2}$ s$^{-1}$

In [ ]:
paoflow.transport(tmin=300.0, tmax=300.0, nt=1, emin=-12.0, emax=2.2, ne=1000, write_to_file=True)
paoflow.finish_execution()

{TODO: from run: the diagonal components of each tensor equal and the off-diagonal ones zero, as the cubic symmetry requires; $S_{xx}$ at $\mu = 0$ (positive, holes) and at the CBM (negative, electrons); $\sigma_{xx}/\tau$ at the VBM}. [Boltzmann transport in silicon](tutorial02.ipynb) plots these tensors, explains how to read them, and replaces the constant $\tau$ with relaxation times from scattering models.

## Where to go next

- [Boltzmann transport in silicon](tutorial02.ipynb) extends this transport calculation to doped Si and to energy-dependent relaxation times.
- [Internal basis workflow](../../internals/internal-basis.md) describes the basis presets of `projections()` and where each shell comes from.
- [The PAO Hamiltonian from projected Kohn–Sham states](../theory/pao-hamiltonian.md) derives the projectability, the PAO shift and the Fourier interpolation, and says what limits them.
- [Band velocities, density of states and adaptive smearing](../theory/density-of-states.md) derives the velocities and the adaptive widths, and lists the sources of error of the DOS.
- [Boltzmann transport in the relaxation-time approximation](../theory/boltzmann-transport.md) derives the transport tensors and says where the constant relaxation time and the rigid bands fail.
- [Advanced modeling of materials with PAOFLOW 2.0: New features and software design](https://doi.org/10.1016/j.commatsci.2021.110828) describes the code.